In [1]:
from climate_cnn import ClimateCNN
from koppen_dataset import KoppenDataset
import torch
import torch.nn as nn
import tensorflow as tf
from torch.utils.data import DataLoader
from sklearn.model_selection import train_test_split
import numpy as np
from torch.amp import autocast
from tqdm.auto import tqdm
from model.dataloader import load_shards

In [2]:
# 1. Is CUDA available?
print(f"Is CUDA available? {torch.cuda.is_available()}")

# 2. Which version of CUDA was PyTorch built with?
print(f"PyTorch CUDA version: {torch.version.cuda}")

# 3. What is the name of your GPU?
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("PyTorch cannot find your GPU.")

Is CUDA available? True
PyTorch CUDA version: 12.8
GPU Name: NVIDIA GeForce RTX 3070


In [3]:
# Load the data from the drive
# Use your local filepath here
file_pattern = r'G:/.shortcut-targets-by-id/1abX3CWvYUSJM3cGg6r_GeHAVeNoYH6Ul/CS6140_Project_Data/koppen_shard_part_*.tfrecord.gz'
all_files = tf.io.gfile.glob(file_pattern)
print(len(all_files), 'shards loaded')

# Split training and testing data
train_files, test_files = train_test_split(all_files, test_size=0.2, random_state=42)

1500 shards loaded


In [4]:
# Function to calculate mean and std of training and testing sets
def calculate_stats(tf_dataset, num_samples=5000):
    images_iterator = tf_dataset.unbatch().take(num_samples).as_numpy_iterator()
    all_images = np.stack([img for img, _ in images_iterator])

    means = np.mean(all_images, axis=(0, 1, 2))
    stds = np.std(all_images, axis=(0, 1, 2))

    return means.tolist(), stds.tolist()

In [5]:
# Load raw data to calculate normalization statistics
loader_batch_size = 512
raw_for_stats = load_shards(train_files, batch_size=512, stats=None)
train_means, train_stds = calculate_stats(raw_for_stats)

# Re-initialize datasets with calculated stats for normalization
train_raw = load_shards(train_files, batch_size=512, stats=(train_means, train_stds))
test_raw = load_shards(test_files, batch_size=512, stats=(train_means, train_stds))

# Create dataloaders
train_loader = DataLoader(KoppenDataset(train_raw), batch_size=None)
test_loader = DataLoader(KoppenDataset(test_raw), batch_size=None)

In [6]:
# Function for checkpointing model during training
def save_checkpoint(state, filename="models/koppen_checkpoint.pth"):
    print(f"=> Saving best model to {filename}")
    torch.save(state, filename)

In [7]:
# Function for loading model checkpoint
def load_checkpoint(checkpoint_path, model, optimizer):
    print(f"=> Loading checkpoint '{checkpoint_path}'")
    checkpoint = torch.load(checkpoint_path)

    model.load_state_dict(checkpoint['state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer'])
    start_epoch = checkpoint['epoch']
    best_acc = checkpoint['best_acc']

    return model, optimizer, start_epoch, best_acc

In [8]:
# Training Loop Function
def train_model(
        model,
        train_dataloader,
        test_dataloader,
        criterion,
        optimizer,
        batch_size=None,
        num_epochs=10,
        start_epoch=0,
        best_val_acc=0,
        patience=5):
    # Set device to CUDA GPU if enabled
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    best_val_acc = best_val_acc
    patience = patience
    epochs_without_improvement = 0
    total_steps = 48000 // batch_size if batch_size is not None else None

    for epoch in range(start_epoch, num_epochs):
        # --- Training ---
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        progress_bar = tqdm(train_dataloader, total=total_steps, desc=f"Epoch {epoch + 1}/{num_epochs} [Train]")

        for images, labels in progress_bar:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            # Forward pass with auto-casting
            with autocast(device_type='cuda', dtype=torch.bfloat16):
                outputs = model(images)
                loss = criterion(outputs, labels)

            # Backward pass with scaled loss
            loss.backward()
            optimizer.step()

            # Statistics
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            # Update progress bar with current memory usage
            mem = torch.cuda.memory_reserved(0) / 1024**2
            progress_bar.set_postfix({"Loss": f"{loss.item():.4f}", "Acc": f"{100.*correct/total:.2f}%", "VRAM": f"{mem:.0f}MB"})

        # --- Validation ---
        model.eval()
        val_correct = 0
        val_total = 0

        with torch.no_grad():
            for images, labels in test_dataloader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, predicted = outputs.max(1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        current_val_acc = 100. * val_correct / val_total

        # Check if this is the best version so far
        if current_val_acc > best_val_acc:
            best_val_acc = current_val_acc
            epochs_without_improvement = 0
            save_checkpoint({
                'epoch': epoch,
                'state_dict': model.state_dict(),
                'optimizer': optimizer.state_dict(),
                'best_acc': best_val_acc,
                'means': train_means,
                'stds': train_stds
            })
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print("Early stopping triggered!")
                break

        print(f"--- Epoch {epoch + 1} Summary: Train Acc: {100. * correct / total:.2f}% | Val Acc: {current_val_acc:.2f}% ---")

In [9]:
# Empty cache from previous runs
torch.cuda.empty_cache()

# Set up the CNN model, optimizer, and loss criterion
cnn = ClimateCNN(num_classes=30)
optim = torch.optim.AdamW(cnn.parameters(), lr=1e-4, weight_decay=1e-2)
loss_func = nn.CrossEntropyLoss(label_smoothing=0.1)

# Train the model
train_model(cnn, train_loader, test_loader, loss_func, optim, batch_size=loader_batch_size)

# Save model weights/biases
torch.save(cnn.state_dict(), 'models/koppen_cnn.pth')

Epoch 1/10 [Train]:   0%|          | 0/93 [00:00<?, ?it/s]

=> Saving best model to models/koppen_checkpoint.pth
--- Epoch 1 Summary: Train Acc: 32.27% | Val Acc: 36.85% ---


Epoch 2/10 [Train]:   0%|          | 0/93 [00:00<?, ?it/s]

=> Saving best model to models/koppen_checkpoint.pth
--- Epoch 2 Summary: Train Acc: 44.83% | Val Acc: 47.23% ---


Epoch 3/10 [Train]:   0%|          | 0/93 [00:00<?, ?it/s]

...

--- Epoch 8 Summary: Train Acc: 63.57% | Val Acc: 51.73% ---


Epoch 9/10 [Train]:   0%|          | 0/93 [00:00<?, ?it/s]

--- Epoch 9 Summary: Train Acc: 66.27% | Val Acc: 50.77% ---


Epoch 10/10 [Train]:   0%|          | 0/93 [00:00<?, ?it/s]

--- Epoch 10 Summary: Train Acc: 70.59% | Val Acc: 52.86% ---


In [10]:
# Quick resume logic
cnn, optim, start, best_accuracy = load_checkpoint("models/koppen_checkpoint.pth", cnn, optim)

# Run for 20 more steps
train_model(cnn, train_loader, test_loader, loss_func, optim, start_epoch=start, num_epochs=30)

=> Loading checkpoint 'models/koppen_checkpoint.pth'


Epoch 8/30 [Train]: 0it [00:00, ?it/s]

=> Saving best model to models/koppen_checkpoint.pth
--- Epoch 8 Summary: Train Acc: 63.56% | Val Acc: 49.14% ---


Epoch 9/30 [Train]: 0it [00:00, ?it/s]

=> Saving best model to models/koppen_checkpoint.pth
--- Epoch 9 Summary: Train Acc: 66.55% | Val Acc: 51.86% ---


...

--- Epoch 13 Summary: Train Acc: 81.85% | Val Acc: 49.48% ---


Epoch 14/30 [Train]: 0it [00:00, ?it/s]

--- Epoch 14 Summary: Train Acc: 84.13% | Val Acc: 50.83% ---


Epoch 15/30 [Train]: 0it [00:00, ?it/s]

Early stopping triggered!
